# 1- Docker MySql

# 🔹 Task 1 – Create test database & employees table (MySQL on Docker)

In [ ]:
docker run -d \
--name mysql-test \
-e MYSQL_ROOT_PASSWORD=password \
-p 3306:3306 \
mysql:8.0


In [ ]:
docker exec -it mysql mysql -u root -p

> password password


In [ ]:
CREATE DATABASE test_db;


In [ ]:
USE test_db;


In [ ]:
CREATE TABLE employees (
    id INT PRIMARY KEY AUTO_INCREMENT,
    name VARCHAR(100),
    department VARCHAR(100),
    salary DECIMAL(10, 2)
);


**Show Databases**

In [ ]:
mysql> show databases;

**Show Tables ?**

In [ ]:
mysql> show tables;

In [ ]:
INSERT INTO employees (name, department, salary) VALUES
('Joe', 'HR', 85000.00),
('Henry', 'Sales', 80000.00),
('Sam', 'Sales', 60000.00),
('Max', 'IT', 90000.00);


In [ ]:
SELECT * FROM employees;


# 2. Import this data using sqoop tool


In [ ]:
docker exec -it sqoop bash

In [ ]:
mkdir test # LOCAL PATH

hdfs dfs -mkdir /test # HDFS PATH

In [ ]:
hdfs dfs -copyFromLocal /dataops/mydata.txt /user/root/inputs/

In [ ]:
export HADOOP_CLASSPATH=/usr/share/java/mysql-connector-java.jar

export HADOOP_CLASSPATH=$HADOOP_CLASSPATH:.

# 3- make sure the data exists in hdfs


In [ ]:
  sqoop eval \
  --connect jdbc:mysql://mysql:3306/test_db \
  --username root \
  --password password \
  --driver com.mysql.jdbc.Driver \
  --query "SELECT * from employees"

In [ ]:
sqoop import \
  -Dmapreduce.framework.name=local \
  --connect jdbc:mysql://mysql:3306/test_db \
  --username root \
  --password password \
  --table employees \
  --target-dir /employees_test_successxbb \
  --delete-target-dir \
  --bindir . \
  -m 1

**To make sure the data is ingested successfully**


In [ ]:
hdfs dfs -cat /employees_test_successxbb/part-m-00000

# 4- use hive to show your data


**Check Mysql ip**

In [ ]:
for i in {2..10}; do (timeout 1 bash -c "cat < /dev/null > /dev/tcp/172.18.0.$i/3306") && echo "MySQL is at 172.18.0.$i" && break; done

**Check name node ip**

In [ ]:
for i in {2..10}; do (timeout 1 bash -c "cat < /dev/null > /dev/tcp/172.18.0.$i/9000") && echo "NameNode (Hadoop) is at 172.18.0.$i" && break; done

**Edit configrution file**

In [ ]:
vi /etc/hosts
172.18.0.3  namenode cluster-master
172.18.0.2  mysql

In [ ]:
DROP TABLE employees_final;

CREATE EXTERNAL TABLE employees_final (
    id INT,
    name STRING,
    dept STRING,
    salary DOUBLE
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION '/employees_test_successxbb'; # hdfs path

**Sqoop Export**

In [ ]:
set hive.exec.mode.local.auto=true;
set hive.vectorized.execution.enabled=true;
set hive.vectorized.execution.reduce.enabled=true;

In [ ]:
CREATE TABLE employees_exported2 (
     id INT ,
  name VARCHAR(100),
  department VARCHAR(100),
  salary DECIMAL(10, 2)
);

In [ ]:
INSERT INTO employees_exported2 (name, department, salary) VALUES
(' Salammox', 'Big DATA', 10000.00);

# 5- export to your database a new data to my sql using sqoop tool

In [ ]:
sqoop export \
  -Dmapreduce.framework.name=local \
  --connect jdbc:mysql://172.18.0.2:3306/test_db \
  --username root \
  --password password \
  --table employees \
  --export-dir /user/hive/warehouse/employees_exported2 \
  --input-fields-terminated-by '\001' \
  --input-null-string '\\N' \
  --input-null-non-string '\\N' \
  --bindir . \
  -m 1